

# Linux Basics for Bioinformatics/Genomics

Prepared By: Udayaraja GK

Supervised By: Haja N Kadarmideen



# Module 2 — Linux / Command Line for Bioinformatics

### Learning objectives
- This explain what the **shell** is and move around the **filesystem** (`pwd`, `ls`, `cd`, `mkdir`), using relative and absolute **paths**;
- match many files at once with **wildcards**, and manage files with `cp`, `mv`, `rm`;
- view and count file contents (`cat`, `head`, `tail`, `wc`);
- pull out columns and rows with `cut`, `grep`, `sort`, `uniq`;
- edit streams with `sed` and filter by column with `awk` — the bioinformatics workhorse;
- chain commands with **pipes** (`|`), save output with **redirection** (`>`), use **variables**, and put a pipeline into a reusable **shell script**.

### How to run this (Google Colab)
1. Open in Colab and **keep the default runtime** — this module uses the shell, **not** R. (No runtime change needed.)
2. **Each code cell begins with `%%bash`**, which runs the whole cell as Linux commands. *(For a single quick command you could instead prefix it with `!`.)*
3. Run cells top to bottom. **Run the "Setup" cell first** — it creates the toy files everything else uses.

Colab's own machine *is* a Linux computer, so these are the exact commands you'd use on a cluster. We work on tiny toy versions of real genomics files: a **VCF** (variants), PLINK **`.bim`/`.fam`** (SNP map / samples), and a **FASTA** (sequence).

## 1. What is the shell?

The **shell** (here **bash**) is a text interface to the computer: you type a command, it runs, it prints the result. Every command follows the pattern `command -options arguments`. This is how bioinformatics files - often too big to open in Excel or load into R - get filtered and reshaped.

## 2. Setup - create the toy files

Run this once. `printf` writes text to a file; `>` sends the output *into* a file (redirection, §12). `\t` is a tab and `\n` a newline.

In [ ]:
%%bash
# A minimal VCF: metadata lines start with ##, the column header with #, then one row per variant.
printf '##fileformat=VCFv4.2\n##source=toy\n#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n15\t44200000\trs_igf1_a\tA\tG\t60\tPASS\tAF=0.30\n15\t44250000\trs_igf1_b\tC\tT\t45\tPASS\tAF=0.12\n15\t44300000\trs_igf1_c\tG\tA\t20\tLowQual\tAF=0.05\n7\t1000000\trs_x\tT\tC\t80\tPASS\tAF=0.40\n7\t2000000\trs_y\tA\tT\t15\tLowQual\tAF=0.02\n1\t5000000\trs_z\tG\tC\t55\tPASS\tAF=0.22\n' > variants.vcf

# PLINK .bim = SNP map (chrom, id, cM, bp, allele1, allele2); .fam = samples (FID IID PID MID SEX PHENO).
printf '15\trs_igf1_a\t0\t44200000\tG\tA\n15\trs_igf1_b\t0\t44250000\tT\tC\n7\trs_x\t0\t1000000\tC\tT\n1\trs_z\t0\t5000000\tC\tG\n' > dogs.bim
printf 'dog1 dog1 0 0 1 30.5\ndog2 dog2 0 0 2 2.5\ndog3 dog3 0 0 1 62\ndog4 dog4 0 0 2 11\n' > dogs.fam

# A tiny FASTA: each sequence starts with a '>' header line.
printf '>chr15_frag\nACGTACGTAC\n>chr7_frag\nTTGGCCAATT\n' > frags.fasta

echo "Files created:"; ls -1

Files created:
dogs.bim
dogs.fam
frags.fasta
sample_data
variants.vcf


## 3. Moving around the filesystem

`pwd` = **p**rint **w**orking **d**irectory (where you are). `ls` = **l**i**s**t files (`-l` = long, with sizes). `mkdir` makes a folder; `cd` changes directory.

In [ ]:
%%bash
pwd                # current location (an ABSOLUTE path, starting from /)
ls -l              # list files here with details
mkdir -p results   # make a folder (-p: no error if it already exists)
ls results         # a RELATIVE path -- looked up from where we are now

/content
total 20
-rw-r--r-- 1 root root   98 Sep 29 09:40 dogs.bim
-rw-r--r-- 1 root root   79 Sep 29 09:40 dogs.fam
-rw-r--r-- 1 root root   45 Sep 29 09:40 frags.fasta
drwxr-xr-x 1 root root 4096 Sep 16 13:26 sample_data
-rw-r--r-- 1 root root  310 Sep 29 09:40 variants.vcf


**Paths:** an **absolute path** starts at the root `/` (e.g. `/content/variants.vcf`) and works from anywhere; a **relative path** (e.g. `results` or `../data`) is read from your current directory. `.` means "here", `..` means "one level up", `~` means your home folder.

> **Two habits that save hours (in a real terminal):** press **Tab** to auto-complete a half-typed file or folder name, and press the **Up arrow** to bring back previous commands (`history` lists them). *(These are interactive, so they don't show in a notebook — but use them everywhere else.)*

## 4. Wildcards - matching many files at once

A **wildcard** stands in for characters, so one command can act on many files. `*` matches any number of characters (`*.vcf` = every file ending in `.vcf`); `?` matches a single character. Essential when a project has hundreds of per-sample files.

In [ ]:
%%bash
ls *.bim *.fam        # every PLINK map and sample file
wc -l *.vcf *.bim     # line counts for all matching files at once

## 5. Looking inside a file

`cat` prints a whole file; `head -n` shows the first *n* lines, `tail -n` the last *n*. For big files never use `cat` — use `head`. (`less` also views files but is interactive, so we skip it in a notebook.)

In [ ]:
%%bash
cat dogs.fam             # small file: print the whole thing (4 samples)

In [ ]:
%%bash
head -4 variants.vcf     # first 4 lines: the metadata + column header + first variant
echo "-----"
tail -2 variants.vcf     # last 2 lines: the final two variants

## 6. Managing files: `cp`, `mv`, `rm`

`cp` copies, `mv` moves *or* renames, `rm` removes. These keep a project tidy — back up a file before editing, move outputs into a results folder.

> ⚠️ **`rm` is permanent** — there is **no Recycle Bin / Trash** on the command line. Double-check the name before you run it, and be very careful combining `rm` with wildcards (`rm *` deletes everything in the folder).

In [ ]:
%%bash
cp variants.vcf variants_backup.vcf   # make a safety copy
mv variants_backup.vcf results/       # move it into results/
ls results/
rm results/variants_backup.vcf        # delete that copy (permanent!)
echo "results/ now contains:"; ls results/

## 7. Counting with `wc`

`wc -l` counts **l**ines. In bioinformatics, lines = records: variants in a VCF, samples in a `.fam`, SNPs in a `.bim`.

In [ ]:
%%bash
wc -l variants.vcf              # total lines (includes header lines!)
grep -vc '^#' variants.vcf      # -v = drop header lines (start with #), -c = count -> real variant count
wc -l dogs.fam                  # number of samples

## 8. Picking columns with `cut`

Files like VCF and `.bim` are **tab-delimited**. `cut -f` selects fields (columns) by number. Here: chromosome, position, and ID (columns 1, 2, 3).

In [ ]:
%%bash
grep -v '^##' variants.vcf | cut -f1,2,3 | head -4    # keep column header (#CHROM), drop ## metadata

## 9. Searching lines with `grep`

**`grep`** prints lines matching a pattern. `^15` means "starts with 15". `-c` counts matches, `-v` inverts (keep non-matching). This is how you pull one chromosome, or strip headers.

In [ ]:
%%bash
grep -c '^15' variants.vcf             # how many variants on chromosome 15?
grep '^15' variants.vcf | cut -f1,2,3  # show them (CHROM POS ID)

> **Exercise 1.** Count how many variants are on chromosome 7. *(Hint: `grep -c '^7' variants.vcf`.)*

## 10. Sorting and de-duplicating: `sort` + `uniq`

`sort` orders lines; `uniq -c` collapses identical adjacent lines and **counts** them. The classic combo `... | sort | uniq -c` tallies categories — here, variants per chromosome.

In [ ]:
%%bash
grep -v '^#' variants.vcf | cut -f1 | sort | uniq -c    # count of variants per chromosome

## 11. Stream editing with `sed`

**`sed`** transforms text as it streams past. `s/find/replace/` substitutes. Here we prepend `chr` to each data line's chromosome (a common format fix), after dropping headers.

In [ ]:
%%bash
grep -v '^#' variants.vcf | sed 's/^/chr/' | cut -f1,2 | head -3

## 12. The workhorse: `awk`

**`awk`** processes a file row by row, with columns available as `$1`, `$2`, … (`$0` is the whole line). You can **filter by a column's value** and **print chosen columns** — exactly what QC and region-extraction need. `-F'\t'` sets the separator to tab; `!/^#/` skips header lines.

In [ ]:
%%bash
# Keep variants with QUAL (column 6) > 30, print CHROM POS ID QUAL
awk -F'\t' '!/^#/ && $6 > 30 {print $1, $2, $3, $6}' variants.vcf

In [ ]:
%%bash
# Region filter: variants on chr15 between 44.1 and 44.26 Mb (around the IGF1 body-size locus)
awk -F'\t' '$1==15 && $2>=44100000 && $2<=44260000 {print $3}' variants.vcf

> **Exercise 2.** Use `awk` to print the ID (`$3`) of every variant whose `FILTER` (column 7) equals `"PASS"`. *(Hint: `$7=="PASS"`.)*

## 13. Pipes and redirection

A **pipe** `|` feeds one command's output into the next (you've used it already). **Redirection** `>` writes output to a file (overwrites); `>>` appends. Together they build one-line pipelines that save results.

In [ ]:
%%bash
# Pipeline: drop headers -> keep PASS variants -> save just their IDs to a file
grep -v '^#' variants.vcf | awk -F'\t' '$7=="PASS" {print $3}' > results/pass_ids.txt
cat results/pass_ids.txt        # show what we saved
wc -l results/pass_ids.txt

## 14. Shell variables

A **variable** stores a value for reuse: `NAME=value` (no spaces around `=`), and you read it back with `$NAME`. Handy for a filename you use repeatedly, so you change it in one place.

In [ ]:
%%bash
VCF=variants.vcf                 # store the filename once
echo "Analyzing $VCF"
grep -vc '^#' "$VCF"             # reuse it (quote "$VCF" to be safe)

## 15. Loops

A **`for` loop** repeats commands over a list — e.g. one step per chromosome or per sample, the backbone of batch processing.

In [ ]:
%%bash
for chr in 1 7 15; do
  n=$(awk -F'\t' -v c="$chr" '!/^#/ && $1==c' variants.vcf | wc -l)   # -v passes the shell var into awk
  echo "chr${chr}: ${n} variants"
done

## 16. Putting it in a script

When a pipeline is worth keeping, save it as a **shell script** (`.sh`) so it is reusable and reproducible. The first line `#!/usr/bin/env bash` (the *shebang*) says "run this with bash". `$1` is the first argument you pass on the command line. Run it with `bash script.sh <file>`.

In [ ]:
%%bash
# Write a small script that counts variants per chromosome in whatever VCF you give it
cat > count_by_chrom.sh <<'EOF'
#!/usr/bin/env bash
VCF=$1                         # first argument = the VCF to read
for chr in 1 7 15; do
  n=$(awk -F'\t' -v c="$chr" '!/^#/ && $1==c' "$VCF" | wc -l)
  echo "chr${chr}: ${n} variants"
done
EOF

bash count_by_chrom.sh variants.vcf    # run it, passing our VCF as $1

## 17. Where this scales (concept)

- **Package/environment managers — conda / mamba.** Bioinformatics tools (`bcftools`, `plink`, `samtools`) are installed into isolated environments, e.g. `mamba install -c bioconda plink2`. This keeps versions reproducible across machines.
- **HPC schedulers — SLURM / PBS.** On a cluster you don't run jobs directly; you *submit* a script (like the one above) with `sbatch count_by_chrom.sh` and the scheduler finds a compute node. The commands inside are the same shell commands you just learned.

The reason this matters: a real VCF can be tens of GB. `grep`/`awk` **stream** it line by line using almost no memory, where loading it whole into R would fail. Shell for wrangling, R for modelling — the two halves of this series.

## Summary

You worked entirely on real bioinformatics file types (VCF, PLINK `.bim`/`.fam`, FASTA) and covered:
**the shell**, filesystem and **paths** (`pwd`, `ls`, `cd`, `mkdir`) · **wildcards** · file management (`cp`, `mv`, `rm`) ·
viewing (`cat`, `head`, `tail`) · counting (`wc`) · columns (`cut`) · searching (`grep`) · tallying (`sort | uniq -c`) ·
stream editing (`sed`) · column filtering (`awk`) · **pipes** (`|`) and **redirection** (`>`) · **variables** ·
**loops** · a reusable **shell script** · and where it scales (conda, SLURM).


### Further reading
- **Data Carpentry — *Introduction to the Command Line for Genomics*** (datacarpentry.github.io/shell-genomics) — the genomics-focused shell course this module draws on.
- *The Unix Workbench* (Sean Kross) — shell fundamentals from scratch.
- `man grep`, `man awk`, `man sed` — the built-in manuals (press `q` to quit).

---